In [ ]:
# install libraries if not already installed
# pip install pandas pandas_datareader fredapi matplotlib seaborn numpy python-dotenv

import pandas as pd
import pandas_datareader as pdr
from fredapi import Fred
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

In [ ]:
# API and Ticker Configuration
# =====================================
# This cell configures the FRED API key, date range, and the tickers for both
# national and city-level data series.

# --- Configuration ---
# Get API key from environment variable
fred_key = os.getenv('FRED_API_KEY')
if not fred_key:
    raise ValueError("FRED_API_KEY not found in environment variables. Please check your .env file.")

fred = Fred(api_key=fred_key)

# Set the desired date range for the analysis

start_date = '1987-01-01'
end_date = '2025-08-01'
var_end_date = '2007-07-01'  # End date for VAR data

In [3]:

# --- Ticker Definitions ---

national_tickers = {
    'cs': 'CSUSHPINSA',  # Case-Shiller National Home Price Index
    'cs10': 'SPCS10RSA',   # Case-Shiller 10-City Home Price Index
    'pce': 'PCEPI',       # Consumer Price Index
    'ffr': 'FEDFUNDS',  # Federal Funds Rate
    'permits': 'PERMIT',      # Building Permits
    'population': 'POPTHM',   # Population in thousands
    'housing starts': 'HOUST',  # Housing Starts
    'recessions': 'USREC',      # Recession Indicator
    'unemp': 'UNRATE'    # Unemployment Rate
}


In [4]:

# Define area-specific tickers for Home Price Index (HPI), Building Permits, and Population

# Case shiller: SF and CE seasonally adjusted
# Permits: SF, CE and CR seasonally adjusted
# Population in thousands
metropol_tickers = {
    
    'SF': {'hpi': 'SFXRSA', 'permits': 'SANF806BPPRIVSA', 'population': 'SFCPOP'},
    'CE': {'hpi': 'CEXRSA', 'permits': 'CLEV439BPPRIVSA', 'population': 'CVLPOP'},
    'CR': {'hpi': 'CRXRNSA', 'permits': 'CHAR737BPPRIVSA', 'population': 'CGRPOP'},
    'LV': {'hpi': 'LVXRNSA', 'permits': 'LASV832BPPRIV', 'population': 'LSVPOP'},
}


In [5]:
# --- Data Download and Separation ---

# Create directory for CSV files
output_dir = 'data_csv'
os.makedirs(output_dir, exist_ok=True)

# Dictionaries to store data
var_data = {}       # For VAR analysis (HPI, Interest rates, etc.) - Full range
visuals_data = {}   # For Visuals (Permits, Population) - 2000-2007


In [6]:

# Date ranges
# start_date and end_date are already defined for VAR data (1987-2007)
visuals_start_date = '2000-01-01'
housing_starts_start_date = '1959-01-01'  # For housing starts data
housing_starts_end_date = '2025-08-01'  # For housing starts data


In [7]:

print("Starting Data Download...")

# 1. Process National Tickers
print("\nProcessing National Data...")
for name, ticker in national_tickers.items():
    try:
        if name in ['permits', 'population']:
            # Visuals data
            series = fred.get_series(ticker, observation_start=visuals_start_date, observation_end=end_date)
            visuals_data[f"National_{name}"] = series
            print(f"  Fetched {name} (Visuals)")
        elif name in ['housing starts', 'recessions']:
            # Visuals data
            series = fred.get_series(ticker, observation_start=housing_starts_start_date, observation_end=housing_starts_end_date)
            visuals_data[f"National_{name}"] = series
            print(f"  Fetched {name} (Visuals)")
        else:
            # VAR data
            series = fred.get_series(ticker, observation_start=start_date, observation_end=var_end_date)
            var_data[name] = series
            print(f"  Fetched {name} (VAR)")
    except Exception as e:
        print(f"  Error fetching {name}: {e}")


Starting Data Download...

Processing National Data...
  Fetched cs (VAR)
  Fetched cs (VAR)
  Fetched cs10 (VAR)
  Fetched cs10 (VAR)
  Fetched pce (VAR)
  Fetched pce (VAR)
  Fetched inflation (VAR)
  Fetched inflation (VAR)
  Fetched ffr (VAR)
  Fetched ffr (VAR)
  Fetched permits (Visuals)
  Fetched permits (Visuals)
  Fetched population (Visuals)
  Fetched population (Visuals)
  Fetched housing starts (Visuals)
  Fetched housing starts (Visuals)
  Fetched recessions (Visuals)
  Fetched recessions (Visuals)
  Fetched unemp (VAR)
  Fetched unemp (VAR)


In [8]:
# 2. Process Metropol Tickers
print("\nProcessing Metropol Data...")
for city, tickers in metropol_tickers.items():
    for var_name, ticker in tickers.items():
        if var_name == 'hpi':
            col_name = city.lower()
        else:
            col_name = f"{city}_{var_name}"

        try:
            if var_name in ['permits', 'population']:
                # Visuals data
                series = fred.get_series(ticker, observation_start=visuals_start_date, observation_end=end_date)
                
                if var_name == 'population':
                    # MSA Population is annual. Resample to monthly and forward fill.
                    series = series.resample('MS').ffill()
                    # Ensure we cover the full range (e.g. if data ends Jan 1st, fill until July)
                    full_range = pd.date_range(start=visuals_start_date, end=end_date, freq='MS')
                    series = series.reindex(full_range, method='ffill')

                visuals_data[col_name] = series
                print(f"  Fetched {col_name} (Visuals)")
            else:
                # VAR data
                series = fred.get_series(ticker, observation_start=start_date, observation_end=var_end_date)
                var_data[col_name] = series
                print(f"  Fetched {col_name} (VAR)")
        except Exception as e:
            print(f"  Error fetching {col_name}: {e}")


Processing Metropol Data...
  Fetched sf (VAR)
  Fetched sf (VAR)
  Fetched SF_permits (Visuals)
  Fetched SF_permits (Visuals)
  Fetched SF_population (Visuals)
  Fetched SF_population (Visuals)
  Fetched ce (VAR)
  Fetched ce (VAR)
  Fetched CE_permits (Visuals)
  Fetched CE_permits (Visuals)
  Fetched CE_population (Visuals)
  Fetched CE_population (Visuals)
  Fetched cr (VAR)
  Fetched cr (VAR)
  Fetched CR_permits (Visuals)
  Fetched CR_permits (Visuals)
  Fetched CR_population (Visuals)
  Fetched CR_population (Visuals)
  Fetched lv (VAR)
  Fetched lv (VAR)
  Fetched LV_permits (Visuals)
  Fetched LV_permits (Visuals)
  Fetched LV_population (Visuals)
  Fetched LV_population (Visuals)


In [9]:

# --- Create DataFrames ---
df_var = pd.DataFrame(var_data)
df_visuals = pd.DataFrame(visuals_data)
df_visuals['National_permits'] *=  1000

# --- Save to CSV ---
var_csv_path = os.path.join(output_dir, 'var_data_1987_2007.csv')
visuals_csv_path = os.path.join(output_dir, 'visuals_data_2000_2007.csv')

df_var.to_csv(var_csv_path)
df_visuals.to_csv(visuals_csv_path)

print(f"\nVAR data saved to {var_csv_path}")
print(f"Visuals data saved to {visuals_csv_path}")

# --- Display Heads ---
print("\nVAR DataFrame Head:")
display(df_var.head())

print("\nVisuals DataFrame Head:")
display(df_visuals.head())


VAR data saved to data_csv\var_data_1987_2007.csv
Visuals data saved to data_csv\visuals_data_2000_2007.csv

VAR DataFrame Head:


,cs,cs10,pce,inflation,ffr,unemp,sf,ce,cr,lv
1987-01-01,63.733,63.103814,51.894,3.16,6.43,6.6,46.955792,54.057393,63.388189,66.358296
1987-02-01,64.132,63.804972,52.076,3.22,6.10,6.6,47.302675,54.229218,63.936295,67.030115
1987-03-01,64.468,64.425674,52.214,3.41,6.13,6.6,47.840213,54.365756,64.171042,67.335161
1987-04-01,64.972,65.063393,52.409,3.57,6.37,6.3,47.984058,54.155370,64.813123,67.882287
1987-05-01,65.547,65.815103,52.542,3.66,6.85,6.3,48.305065,54.726192,65.181595,67.898858



Visuals DataFrame Head:


,National_permits,National_population,National_housing starts,National_recessions,SF_permits,SF_population,CE_permits,CE_population,CR_permits,CR_population,LV_permits,LV_population
1959-01-01,NaN,NaN,1657.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-02-01,NaN,NaN,1667.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-03-01,NaN,NaN,1620.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-04-01,NaN,NaN,1590.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-05-01,NaN,NaN,1498.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
